In [38]:
import pandas as pd 
import requests, json


In [39]:
# Load full dataset with predictions
energy_df = pd.read_csv("../data/energy_source_with_predictions.csv")
# Load test results with errors
rf_results = pd.read_csv("../data/rf_results.csv")

In [40]:
# Create a summaary to review the structure of the dataset
summary = pd.DataFrame({
    'Unique Values': energy_df.nunique(),
    'Data Type': energy_df.dtypes,
    'Missing Values (Total)': energy_df.isnull().sum(),
    'Missing Values (%)': (energy_df.isnull().sum() / len(energy_df)) * 100
})
# Sort by percentage of missing values
summary = summary.sort_values(by='Missing Values (%)', ascending=False)

print(summary)

                        Unique Values Data Type  Missing Values (Total)  \
Date                             2922    object                       0   
region_key                         12    object                       0   
ECLAIR                             56   float64                       0   
TX                              31040   float64                       0   
TNSOL                           24575   float64                       0   
FXY                             24707   float64                       0   
DRR                             13292   float64                       0   
year                                8     int64                       0   
population                         91   float64                       0   
FUMEE                              24   float64                       0   
density                            91   float64                       0   
superf                             12   float64                       0   
Consumption              

In [41]:
# Create a summaary to review the structure of the dataset
summary = pd.DataFrame({
    'Unique Values': rf_results.nunique(),
    'Data Type': rf_results.dtypes,
    'Missing Values (Total)': rf_results.isnull().sum(),
    'Missing Values (%)': (rf_results.isnull().sum() / len(rf_results)) * 100
})
# Sort by percentage of missing values
summary = summary.sort_values(by='Missing Values (%)', ascending=False)

print(summary)

              Unique Values Data Type  Missing Values (Total)  \
Date                   2655    object                       0   
region_name              12    object                       0   
abs_error_rf           6348   float64                       0   
error_rf               6348   float64                       0   
y_pred_rf              6348   float64                       0   
y_true                 6344   float64                       0   
day_of_week               7     int64                       0   
Workday?                  2     int64                       0   
density                  89   float64                       0   
year                      8     int64                       0   
DRR                    3844   float64                       0   
FXY                    5935   float64                       0   
TNSOL                  5922   float64                       0   
TX                     6319   float64                       0   
ECLAIR                   

In [42]:
print(energy_df["region_name"].unique())

['Auvergne-Rhône-Alpes' 'Bourgogne-Franche-Comté' 'Bretagne'
 'Centre-Val de Loire' 'Grand Est' 'Hauts-de-France' 'Normandie'
 'Nouvelle-Aquitaine' 'Occitanie' 'Pays de la Loire'
 "Provence-Alpes-Côte d'Azur" 'Île-de-France']


### Methodological note — error metrics

The **Absolute Percentage Error (APE)** is computed at the observation level as:

$$APE = \\frac{|y_{true} - y_{pred}|}{y_{true} + \\epsilon}$$

where $\\epsilon = 10^{-8}$ is a small constant added to avoid division by zero. 
Its effect is negligible given that consumption values are in the tens of kWh per capita.

**Example:** for $y_{true} = 40.0$ and $y_{pred} = 42.0$:

| Metric | Value |
|:-------|------:|
| Error ($y_{true} - y_{pred}$) | −2.0 |
| Absolute error | 2.0 |
| APE | 0.05 → 5% |

The **regional MAPE** is the mean of all individual APEs for test observations 
belonging to that region:

$$MAPE_{region} = \\frac{1}{n} \\sum_{i=1}^{n} APE_i$$

All error metrics are computed **on the held-out test set only** (`rf_results` derives 
from `X_test` / `y_test`). This ensures we measure generalization to unseen data, 
not in-sample fit.

In [43]:
tooltip_data = rf_results.groupby('region_name').agg(
    mean_mape      = ('ape_rf',       'mean'),
    mean_mae       = ('abs_error_rf', 'mean'),
    mean_predicted = ('y_pred_rf',    'mean'),
    mean_true      = ('y_true',       'mean'),
    mean_bias      = ('error_rf',     'mean'),
    density        = ('density',      'mean'),
    n_obs          = ('y_true',       'count')
).round(3).reset_index()

# Convert to dict keyed by region name — easy to look up in D3
metrics_dict = tooltip_data.set_index('region_name').to_dict(orient='index')

with open('../data/region_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics_dict, f, ensure_ascii=False, indent=2)

print(json.dumps(metrics_dict, ensure_ascii=False, indent=2))

{
  "Auvergne-Rhône-Alpes": {
    "mean_mape": 0.04,
    "mean_mae": 1.8,
    "mean_predicted": 45.427,
    "mean_true": 45.352,
    "mean_bias": -0.076,
    "density": 112.586,
    "n_obs": 584
  },
  "Bourgogne-Franche-Comté": {
    "mean_mape": 0.051,
    "mean_mae": 2.002,
    "mean_predicted": 41.075,
    "mean_true": 40.954,
    "mean_bias": -0.121,
    "density": 58.586,
    "n_obs": 523
  },
  "Bretagne": {
    "mean_mape": 0.045,
    "mean_mae": 1.609,
    "mean_predicted": 36.954,
    "mean_true": 36.506,
    "mean_bias": -0.448,
    "density": 121.641,
    "n_obs": 569
  },
  "Centre-Val de Loire": {
    "mean_mape": 0.049,
    "mean_mae": 1.914,
    "mean_predicted": 39.756,
    "mean_true": 39.849,
    "mean_bias": 0.093,
    "density": 65.3,
    "n_obs": 324
  },
  "Grand Est": {
    "mean_mape": 0.042,
    "mean_mae": 1.797,
    "mean_predicted": 44.569,
    "mean_true": 44.519,
    "mean_bias": -0.05,
    "density": 96.29,
    "n_obs": 581
  },
  "Hauts-de-France": {
  

In [44]:
viz_b_data = rf_results[['y_true', 'y_pred_rf', 'abs_error_rf', 'ape_rf', 'region_name']].copy()
viz_b_data = viz_b_data.round(3)

viz_b_data.to_json('../data/rf_scatter.json', orient='records')
print(f"Total points: {len(viz_b_data)}")
print(f"y_true:  {viz_b_data['y_true'].min():.2f} — {viz_b_data['y_true'].max():.2f}")
print(f"y_pred:  {viz_b_data['y_pred_rf'].min():.2f} — {viz_b_data['y_pred_rf'].max():.2f}")
print(f"n regions: {viz_b_data['region_name'].nunique()}")

Total points: 6348
y_true:  18.96 — 74.95
y_pred:  21.11 — 71.49
n regions: 12


In [45]:
from scipy import stats
import pandas as pd

# ANOVA
groups = [group['ape_rf'].values for _, group in rf_results.groupby('region_name')]
f_stat, p_value = stats.f_oneway(*groups)

# Per-region stats
anova_table = rf_results.groupby('region_name').agg(
    n        = ('ape_rf', 'count'),
    mean_mape= ('ape_rf', 'mean'),
    std_mape = ('ape_rf', 'std'),
    min_mape = ('ape_rf', 'min'),
    max_mape = ('ape_rf', 'max'),
).round(4).reset_index()

anova_table = anova_table.sort_values('mean_mape', ascending=False)

print(f"F-statistic: {f_stat:.3f}")
print(f"p-value:     {p_value:.2e}")
print(f"\n{anova_table.to_string(index=False)}")

F-statistic: 10.029
p-value:     2.36e-18

               region_name   n  mean_mape  std_mape  min_mape  max_mape
             Île-de-France 532     0.0514    0.0445    0.0001    0.2437
   Bourgogne-Franche-Comté 523     0.0513    0.0509    0.0002    0.4059
       Centre-Val de Loire 324     0.0490    0.0394    0.0005    0.2169
                  Bretagne 569     0.0448    0.0476    0.0000    0.3812
                 Grand Est 581     0.0417    0.0421    0.0001    0.3177
                 Occitanie 595     0.0417    0.0391    0.0001    0.2434
          Pays de la Loire 327     0.0417    0.0371    0.0004    0.2350
                 Normandie 599     0.0416    0.0347    0.0000    0.2195
           Hauts-de-France 538     0.0408    0.0354    0.0001    0.2740
      Auvergne-Rhône-Alpes 584     0.0401    0.0351    0.0001    0.2062
        Nouvelle-Aquitaine 569     0.0367    0.0343    0.0001    0.2171
Provence-Alpes-Côte d'Azur 607     0.0337    0.0306    0.0001    0.2042


In [46]:
anova_data = {
    "f_stat": round(f_stat, 3),
    "p_value": 2.36e-18,
    "regions": anova_table.rename(columns={
        'region_name': 'name',
        'n':           'n',
        'mean_mape':   'mean',
        'std_mape':    'std',
        'min_mape':    'min',
        'max_mape':    'max'
    }).to_dict(orient='records')
}

with open('../data/anova_results.json', 'w', encoding='utf-8') as f:
    json.dump(anova_data, f, ensure_ascii=False, indent=2)

print("Saved.")

Saved.


In [47]:
import json

violin_data = rf_results[['region_name', 'error_rf', 'ape_rf']].copy()
violin_data = violin_data.round(4)

# Order regions by median MAPE (worst to best) for the chart
region_order = (rf_results.groupby('region_name')['ape_rf']
                .median()
                .sort_values(ascending=False)
                .index.tolist())

print("Region order (worst to best median MAPE):")
print(region_order)

print(f"\nTotal rows: {len(violin_data)}")
print(f"error_rf range: {violin_data['error_rf'].min():.3f} — {violin_data['error_rf'].max():.3f}")
print(f"ape_rf range:   {violin_data['ape_rf'].min():.4f} — {violin_data['ape_rf'].max():.4f}")

violin_data.to_json('../data/violin_data.json', orient='records')
print("\nSaved.")

Region order (worst to best median MAPE):
['Centre-Val de Loire', 'Île-de-France', 'Bourgogne-Franche-Comté', 'Normandie', 'Pays de la Loire', 'Hauts-de-France', 'Occitanie', 'Bretagne', 'Auvergne-Rhône-Alpes', 'Grand Est', 'Nouvelle-Aquitaine', "Provence-Alpes-Côte d'Azur"]

Total rows: 6348
error_rf range: -13.590 — 9.993
ape_rf range:   0.0000 — 0.4059

Saved.


In [49]:
viz_e_data = rf_results.groupby('region_name').agg(
    mean_mape = ('ape_rf', 'mean'),
    std_mape  = ('ape_rf', 'std'),
    density   = ('density', 'mean'),
).round(4).reset_index()

print(viz_e_data.sort_values('density', ascending=False).to_string(index=False))
viz_e_data.to_json('../data/viz_e_data.json', orient='records')
print("\nSaved.")


               region_name  mean_mape  std_mape   density
             Île-de-France     0.0514    0.0445 1009.7952
           Hauts-de-France     0.0408    0.0354  187.5871
Provence-Alpes-Côte d'Azur     0.0337    0.0306  159.4117
                  Bretagne     0.0448    0.0476  121.6410
          Pays de la Loire     0.0417    0.0371  115.4182
      Auvergne-Rhône-Alpes     0.0401    0.0351  112.5864
                 Normandie     0.0416    0.0347  110.5998
                 Grand Est     0.0417    0.0421   96.2902
                 Occitanie     0.0417    0.0391   79.9892
        Nouvelle-Aquitaine     0.0367    0.0343   70.1625
       Centre-Val de Loire     0.0490    0.0394   65.3002
   Bourgogne-Franche-Comté     0.0513    0.0509   58.5862

Saved.
